In [ ]:
# | default_exp wuilt.client

# Wuilt Client
> GraphQL client for the Wuilt e-commerce API.

In [ ]:
# | export
import httpx
import time

In [ ]:
# | export
WUILT_GRAPHQL_URL = "https://graphql.wuilt.com"

In [ ]:
# | export
class WuiltClient:
    "GraphQL client for Wuilt store API. Handles auth, requests, and rate limiting."

    def __init__(self, api_key: str, store_id: str, locale: str = "ar"):
        self.api_key = api_key
        self.store_id = store_id
        self.locale = locale
        self._headers = {"X-API-KEY": api_key}
        self._last_request = 0.0
        self._min_interval = 1.2

    def _request(self, query: str, variables: dict | None = None) -> dict:
        "Send a GraphQL request with rate limiting."
        elapsed = time.time() - self._last_request
        if elapsed < self._min_interval:
            time.sleep(self._min_interval - elapsed)
        try:
            response = httpx.post(
                WUILT_GRAPHQL_URL,
                json={"query": query, "variables": variables or {}},
                headers=self._headers,
                timeout=30,
            )
            response.raise_for_status()
            data = response.json()
            self._last_request = time.time()
            errors = data.get("errors")
            if errors:
                msg = errors[0].get("message", str(errors))
                if "Rate limit" in msg:
                    time.sleep(5)
                    return self._request(query, variables)
                raise RuntimeError(f"Wuilt API error: {msg}")
            return data["data"]
        except httpx.HTTPError as e:
            raise ConnectionError(f"Wuilt API request failed: {e}")

    def get_store(self) -> dict:
        "Fetch store info with SEO and robots.txt fields."
        query = """
        query GetStore($id: ID!) {
            store(id: $id) {
                id name
                seo { title description }
                robotsTxt { id content }
            }
        }
        """
        result = self._request(query, {"id": self.store_id})
        return result.get("store") or {}

    def update_store_seo(
        self,
        seo_title: str | None = None,
        seo_description: str | None = None,
        name: str | None = None,
    ) -> dict:
        "Update store-level SEO fields. Requires admin API key."
        mutation = """
        mutation UpdateStore($input: UpdateStoreInput!) {
            updateStore(input: $input) {
                store { id name seo { title description } }
            }
        }
        """
        inp = {"storeId": self.store_id}
        if seo_title is not None or seo_description is not None:
            seo = {}
            if seo_title is not None:
                seo["title"] = seo_title
            if seo_description is not None:
                seo["description"] = seo_description
            inp["seo"] = seo
        if name is not None:
            inp["name"] = name
        result = self._request(mutation, {"input": inp})
        return result["updateStore"]["store"]

    def get_products(self) -> list[dict]:
        "Fetch all products with SEO fields, images, and variants."
        query = """
        query ListStoreProducts($filter: ProductsFilterInput) {
            products(filter: $filter) {
                totalCount
                nodes {
                    id title handle
                    descriptionHtml shortDescription
                    isVisible status
                    seo { title description }
                    images { id src originalSrc altText status }
                    variants(first: 10) {
                        nodes { id sku price { amount currencyCode } quantity }
                    }
                }
            }
        }
        """
        result = self._request(query, {"filter": {"storeIds": [self.store_id]}})
        return result["products"]["nodes"]

    def get_product(self, product_id: str) -> dict | None:
        "Fetch a single product by ID."
        query = """
        query GetProduct($storeId: ID!, $id: ID!) {
            product(storeId: $storeId, id: $id) {
                id title handle
                descriptionHtml shortDescription
                isVisible status
                seo { title description }
                images { id src originalSrc altText status }
                variants { id sku price { amount currencyCode } quantity }
            }
        }
        """
        result = self._request(query, {"storeId": self.store_id, "id": product_id})
        return result.get("product")

    def update_product_seo(
        self,
        product_id: str,
        seo_title: str | None = None,
        seo_description: str | None = None,
        description_html: str | None = None,
        short_description: str | None = None,
    ) -> dict:
        "Update a product's SEO fields. Only non-None fields are sent."
        mutation = """
        mutation UpdateProduct($input: ProductInput!, $locale: String!) {
            updateProduct(input: $input, locale: $locale) {
                product { id title seo { title description } }
            }
        }
        """
        inp = {"id": product_id, "storeId": self.store_id}
        seo = {}
        if seo_title is not None:
            seo["title"] = seo_title
        if seo_description is not None:
            seo["description"] = seo_description
        if seo:
            inp["seo"] = seo
        if description_html is not None:
            inp["descriptionHtml"] = description_html
        if short_description is not None:
            inp["shortDescription"] = short_description
        result = self._request(mutation, {"input": inp, "locale": self.locale})
        return result["updateProduct"]["product"]

    def get_collections(self) -> list[dict]:
        "Fetch all collections with SEO fields."
        query = """
        query GetCollections($filter: ProductCollectionFilterInput) {
            collections(filter: $filter, connection: { first: 50 }) {
                totalCount
                nodes {
                    id title handle
                    descriptionHtml isVisible
                    seo { title description }
                    image { id src altText }
                }
            }
        }
        """
        result = self._request(query, {"filter": {"storeId": self.store_id}})
        return result["collections"]["nodes"]

    def update_collection_seo(
        self,
        collection_id: str,
        seo_title: str | None = None,
        seo_description: str | None = None,
        description_html: str | None = None,
    ) -> dict:
        "Update a collection's SEO fields."
        mutation = """
        mutation UpdateCollection($input: CollectionInput!, $locale: String!) {
            updateCollection(input: $input, locale: $locale) {
                collection { id title seo { title description } }
            }
        }
        """
        inp = {"id": collection_id, "storeId": self.store_id}
        seo = {}
        if seo_title is not None:
            seo["title"] = seo_title
        if seo_description is not None:
            seo["description"] = seo_description
        if seo:
            inp["seo"] = seo
        if description_html is not None:
            inp["descriptionHtml"] = description_html
        result = self._request(mutation, {"input": inp, "locale": self.locale})
        return result["updateCollection"]["collection"]

    def get_pages(self) -> list[dict]:
        "Fetch all store pages with SEO fields."
        query = """
        query GetStorePages($storeId: ID!) {
            storePages(storeId: $storeId, filter: {}, connection: { first: 50 }) {
                totalCount
                nodes {
                    id name handle
                    status pageType locale
                    seo { title description }
                }
            }
        }
        """
        result = self._request(query, {"storeId": self.store_id})
        return result["storePages"]["nodes"]

    def get_page(self, page_id: str) -> dict | None:
        "Fetch a single store page by ID."
        query = """
        query GetStorePage($storeId: ID!, $id: ID!) {
            storePage(storeId: $storeId, id: $id) {
                id name handle
                status pageType locale
                seo { title description }
                translations { locale name seo { title description } }
            }
        }
        """
        result = self._request(query, {"storeId": self.store_id, "id": page_id})
        return result.get("storePage")

    def update_page_seo(
        self,
        page_id: str,
        name: str | None = None,
        seo_title: str | None = None,
        seo_description: str | None = None,
    ) -> dict:
        "Update a page's SEO fields. Returns the StorePage directly."
        mutation = """
        mutation UpdateStorePage($storeId: ID!, $id: ID!, $input: UpdateStorePageInput!) {
            updateStorePage(storeId: $storeId, id: $id, input: $input) {
                id name seo { title description }
            }
        }
        """
        inp = {}
        if name is not None:
            inp["name"] = name
        if seo_title is not None or seo_description is not None:
            seo = {}
            if seo_title is not None:
                seo["title"] = seo_title
            if seo_description is not None:
                seo["description"] = seo_description
            inp["seo"] = seo
        result = self._request(mutation, {
            "storeId": self.store_id, "id": page_id, "input": inp,
        })
        return result["updateStorePage"]

    def create_page(
        self,
        name: str,
        handle: str = "",
        page_type: str = "CUSTOM",
        locale: str | None = None,
        seo_title: str | None = None,
        seo_description: str | None = None,
    ) -> dict:
        "Create a new store page."
        mutation = """
        mutation CreateStorePage($storeId: ID!, $input: CreateStorePageInput!) {
            createStorePage(storeId: $storeId, input: $input) {
                id name handle seo { title description }
            }
        }
        """
        inp = {"name": name, "locale": locale or self.locale}
        if handle:
            inp["handle"] = handle
        inp["pageType"] = page_type
        if seo_title is not None or seo_description is not None:
            seo = {}
            if seo_title is not None:
                seo["title"] = seo_title
            if seo_description is not None:
                seo["description"] = seo_description
            inp["seo"] = seo
        result = self._request(mutation, {"storeId": self.store_id, "input": inp})
        return result["createStorePage"]

    def delete_page(self, page_id: str) -> bool:
        "Delete a store page. Returns True on success."
        mutation = """
        mutation DeleteStorePage($storeId: ID!, $id: ID!) {
            deleteStorePage(storeId: $storeId, id: $id)
        }
        """
        result = self._request(mutation, {"storeId": self.store_id, "id": page_id})
        return result["deleteStorePage"]

    def get_robots_txt(self) -> str:
        "Fetch the store's robots.txt content via store query."
        store = self.get_store()
        robots = store.get("robotsTxt") or {}
        return robots.get("content") or ""

    def update_robots_txt(self, content: str) -> bool:
        "Update the store's robots.txt content. Returns True if successful."
        mutation = """
        mutation UpdateStoreRobotsTxt($storeId: ID!, $content: String!) {
            updateStoreRobotsTxt(storeId: $storeId, content: $content)
        }
        """
        result = self._request(mutation, {"storeId": self.store_id, "content": content})
        return result["updateStoreRobotsTxt"]

In [ ]:
# | hide
# | eval: false
from dotenv import load_dotenv

load_dotenv()
import os

WUILT_STORE_ID = os.getenv("WUILT_STORE_ID")
WUILT_API_KEY = os.getenv("WUILT_API_KEY")
client = WuiltClient(
    api_key=WUILT_API_KEY,
    store_id=WUILT_STORE_ID,
    locale="ar",
)

# Store
store = client.get_store()
print(f"Store: {store.get('name')}")
print(f"  SEO: {store.get('seo')}")
print(f"  Robots: {bool(store.get('robotsTxt'))}")

# Products
products = client.get_products()
print(f"Products: {len(products)}")

# Collections
collections = client.get_collections()
print(f"Collections: {len(collections)}")

# Pages
pages = client.get_pages()
print(f"Pages: {len(pages)}")